# 07. 폐루프와 실행 길이 비교 / 90분

**강사 설명 → 완성된 셀 실행 → 조건 비교 → 결과 해설** 순서입니다. 코드 작성 과제나 관찰 답안 제출은 없습니다.

밖의 반복문은 재관측과 추론, 안의 반복문은 행동 묶음 일부의 적용입니다.

[단원 실행 안내](README.md) · [슬라이드 실습 시작](https://docs.google.com/presentation/d/1yAFoaiH15_qTveoSi5r1gF1TPE7bFf2CrLVLUolsg4k/edit#slide=id.d5_lab07_start)

각 단원은 새 커널에서 시작합니다. 노트북 셀은 위에서 아래로 실행하고, `bash` 명령은 지정한 터미널에서 실행합니다. 결과가 이미 있으면 재사용 여부와 실행 이름을 확인합니다.


## 1. 아키텍처 — 지금 실행하는 부분

![폐루프와 실행 길이 비교의 입력·처리·출력](../교재그림/guided-vla/day5-07.svg)

while은 새 batch를 만들어 다음 chunk를 예측합니다. for는 남은 스텝과 execute 중 작은 수만큼 행동을 적용합니다. 매 step에서 성공·종료를 검사하며 다음 추론 전에 reset하지 않습니다.

**그림을 읽는 순서:** 1→2→3 뒤 아래 4→5→6으로 이어집니다. 실시간 반복은 새 관측이 다음 입력으로 들어오는 과정이며, 저장 결과를 읽는 셀은 모델을 다시 실행하지 않습니다.


### 전체 실행 지도에서 현재 단원 찾기

![5일차 실행 경계 전체 지도](../교재그림/guided-vla/detailed/day5-system.svg)

브라우저는 실행을 요청하고 결과를 표시합니다. 커널이 호출한 별도 runner가 물리 환경과 정책을 연결합니다. 01·02의 합성 셀은 구조 설명용이고 실제 환경 실행은 04, 모델 실행은 06부터입니다. 아래 단원별 도식에서 이번에 실제로 실행할 범위를 확인합니다.

## 상세 해설 — 앞선 수업 없이도 흐름을 읽기

아래 설명은 1~3일차에서 배운 개념도 이번 실습의 입력·코드·출력에 맞춰 다시 풉니다. 실행 중에는 필요한 부분을 읽고, 수업 뒤에는 독립적인 참고 자료로 사용할 수 있습니다. 추가 작성 과제는 없습니다.

### 1. 기초 복습: 폐루프와 실행 horizon

- 폐루프는 행동 결과를 관측해 다음 행동에 반영합니다.
- chunk 길이50과 execute=5/10은 다른 설정입니다.
- 짧은 execute는 더 자주 재관측·추론할 기회를 만듭니다.
- 호출 비용과 피드백 빈도를 함께 비교합니다.

개방형 실행은 처음 만든 계획을 그대로 적용하는 방향이고, 폐루프는 실행 후 상태를 다시 읽어 다음 행동을 정합니다. 이 runner는 한 chunk의 앞부분을 적용한 다음 새 obs로 정책을 다시 호출합니다. chunk 안에서는 여러 행동을 연속 적용하고 chunk 사이에서는 새 관측을 반영합니다.

execute=10이면 한 예측에서 최대 10개, 5이면 최대 5개를 적용합니다. 두 경우 모두 모델은 50개를 예측합니다. 짧은 execute는 새 관측으로 다시 판단할 기회를 늘리지만 동일 스텝을 진행하는 데 추론 호출이 더 많이 필요할 수 있습니다. 더 짧다고 무조건 성공률이 높거나 총 시간이 짧아지는 것은 아닙니다.

이 수업의 동기 실행에서는 추론 동안 환경이 대기합니다. 따라서 피드백 빈도, 시뮬레이션 스텝 수, 벽시계 실행 시간을 서로 같은 주파수로 해석하지 않습니다. 기록된 영상과 시간 지표를 함께 비교합니다.


### 2. 실행 아키텍처: 바깥 while과 안쪽 for

![실행 주체와 데이터 경계](../교재그림/guided-vla/detailed/day5-07.svg)

- while: 현재 obs → batch → 새 chunk 예측
- for: 앞 min(execute, 남은 스텝)개 적용
- 매 step: obs 갱신 → 성공·종료 검사
- 계속하면 새 obs로 while 복귀; reset하지 않음

바깥 while은 남은 실행 한도가 있는 동안 새 관측으로 chunk를 예측합니다. 안쪽 for는 chunk[:min(execute,args.steps-result[steps])]에서 행동을 꺼냅니다. 남은 한도가 3이면 execute=10이어도 세 행동만 적용하므로 마지막 묶음이 항상 같은 길이는 아닙니다.

env.step은 obs, reward, terminated, truncated, info를 반환합니다. obs를 다음 상태로 갱신하고 steps를 하나 증가시킨 뒤 is_success를 확인합니다. 성공하거나 terminated/truncated이면 안쪽 반복을 멈추고 바깥 반복도 끝납니다. reward 하나만 보고 성공이라고 판정하지 않습니다.

한 chunk의 앞10개를 적용한 뒤 나머지40개는 다음 실행 목록으로 이어 쓰지 않습니다. 새 관측에서 다시 예측합니다. 이때 env.reset을 넣으면 진행한 물체 상태가 사라지고 매 묶음마다 시작 장면으로 돌아가므로 폐루프의 의미가 깨집니다.


### 3. 코드 추적: 스텝·추론 횟수·종료

- 최대280스텝, 조기 종료가 없으면 h10은 28회 추론
- 같은 가정에서 h5는 56회 추론이 필요합니다.
- 실제 횟수는 성공·종료·중단 여부에 따라 달라집니다.
- 각 행동은 clip 후 적용; 성공 정보가 없으면 오류

280을 10으로 나누면 28, 5로 나누면 56입니다. 이는 모든 예측이 정상이고 조기 종료 없이 스텝 한도까지 갔다는 계산 예시입니다. 실제 로그에서 성공이 일찍 나오면 steps와 chunk 수가 더 작아집니다. 따라서 추론 횟수가 적다고 효율만 좋아졌다고 결론 내리기 전에 종료 이유를 읽습니다.

for 안의 clip은 행동의 각 성분을 −1~1로 제한합니다. 모든 성분을 0으로 만든 벡터를 실물 로봇의 보편적인 정지 명령이라고 일반화하지 않습니다. 행동 의미는 환경 제어 인터페이스에 달려 있습니다. 이 수업은 turtlesim Twist와 다른 행동 계약을 사용합니다.

info에 is_success가 없으면 KeyError를 발생시킵니다. 환경 어댑터가 바뀌었는데 성공 정보가 사라진 경우를 조용히 실패로 집계하지 않기 위해서입니다. step_limit으로 정상 종료된 completed와 예외로 중단된 incomplete는 결과 상태가 다릅니다.


### 4. h10/h5 비교와 직접 수정할 부분

- 과제·초기0·seed7·최대280·모델을 고정합니다.
- 영상·steps·chunk 수·chunk_seconds·성공을 함께 봅니다.
- 기존 결과는 재사용하며 재측정은 새 이름으로 남깁니다.
- 반복문 수정 시 남은 한도와 조기 종료를 보존합니다.

기준 seed7-h10과 비교 seed7-h5의 차이는 execute 하나입니다. 초기 배치나 모델도 바뀌면 결과 차이를 horizon 때문이라고 설명할 수 없습니다. 한 조건 한 번의 실행은 탐색 비교이며 일반적인 우열을 보장하지 않습니다. 성공 여부뿐 아니라 어디서 접근·접촉·그리퍼 동작이 달라지는지 영상을 함께 읽습니다.

execution_seconds에는 추론 외 행동 적용·기록·화면 갱신 등 실행 비용이 들어갈 수 있습니다. chunk_seconds의 합과 전체 시간이 같지 않다고 곧바로 측정 오류는 아닙니다. 영상 fps=20 또한 별도 재생 설정입니다. 시간 지표마다 포함 구간을 코드와 연결해 설명합니다.

직접 루프를 수정한다면 args.execute 유효 범위, 남은 스텝 계산, 매 step의 success 검사, done 후 바깥 break를 함께 확인합니다. 추론을 비동기로 확장하는 것은 단순 for문 변경이 아니라 관측·행동의 시점과 오래된 계획의 처리 규칙까지 새로 설계하는 일입니다.


## 2. 코드 위치와 변경 범위

| 구분 | 이 단원에서 확인할 내용 |
|---|---|
| 실제 소스 | [scripts/run_episode.py : while / for](scripts/run_episode.py) |
| 바꾸어 볼 조건 | `execute=10 / 5, 초기0·seed7 고정` |
| 예상되는 차이 | seed7-h10과 seed7-h5를 차례로 실행해 영상·스텝·추론 횟수·시간을 비교합니다. 한 조건 한 번으로 우열을 일반화하지 않습니다. |
| 직접 구현·수정할 때 | execute는 1~50 범위입니다. 같은 과제·seed·초기상태·최대280스텝을 유지해야 horizon의 영향을 비교할 수 있습니다. 0 벡터를 실물 로봇의 정지로 가정하지 않습니다. |

설정값 변경은 아래 준비된 비교 실행으로 진행합니다. 소스를 직접 변경하는 경우에는 실행 중인 프로세스를 종료하고 다시 실행해야 합니다. ROS 패키지 파일은 재빌드·환경 source까지 확인합니다.


### 실제 코드 읽기 — 입력·처리·출력 연결

while은 새 batch를 만들어 다음 chunk를 예측합니다. for는 남은 스텝과 execute 중 작은 수만큼 행동을 적용합니다. 매 step에서 성공·종료를 검사하며 다음 추론 전에 reset하지 않습니다.


In [ ]:
from pathlib import Path
from IPython.display import Code, display
import ast
source_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'10.VLA_v2/scripts').is_dir())
source_path = source_root/'10.VLA_v2'/'scripts/run_episode.py'
text = source_path.read_text()
# 함수 위치를 AST로 찾아 해당 함수 전체를 표시합니다. 파일을 import하거나 모델을 로드하지 않습니다.
functions = [node for node in ast.parse(text).body if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))]
for node in functions:
    if node.name in ['run']:
        print(source_path, '함수:', node.name, '줄:', node.lineno)
        display(Code('\n'.join(text.splitlines()[node.lineno-1:node.end_lineno]), language='python'))


## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [5일차 실행 안내](README.md)를 참고합니다.

**5일차 환경:** `nepes-course:physical-ai-vla-v1.0`, Python `/opt/course/bin/python`을 사용합니다. 같은 날에는 준비된 개인 VLA 환경을 유지합니다.

**실행 전 확인 — 컨테이너 내부 Terminal:**

```bash
cd /workspace
/opt/course/bin/python 10.VLA_v2/scripts/preflight.py
```

Python 커널도 `/opt/course/bin/python`을 사용합니다. 강사 배포 `Data/models/smolvla-libero-probe`(LIBERO용 정책 revision `31d453f7edd78c839a8bbc39744a292686daf0de`), `Data/cache/smolvla-probe/huggingface`, `Data/cache/smolvla-probe/libero`와 LIBERO 자산이 필요합니다. 파일 누락은 호스트 개인 Data의 마운트·배포본을 확인합니다. 수강생이 `pip install`, Docker build, 모델 다운로드를 하지 않습니다. CUDA/EGL 문제는 전용 이미지·NVIDIA runtime과 다른 GPU 작업을 확인합니다.

**실행과 저장:** 04부터 08까지의 노트북의 `SESSION = 'v2-session-01'`은 같은 회차이며 결과는 `Data/logs/day5/v2-session-01/<실행이름>/`에 남습니다. 새 비교 전체는 모든 후속 노트북의 SESSION을 함께 `session-02` 등으로 바꿉니다. `run_lesson`은 동일 조건의 `completed` 결과를 읽고 없을 때 실행합니다. 미완료·조건 불일치는 기존 파일을 삭제하거나 성공값으로 바꾸지 말고 새 회차/실행 이름을 사용합니다. JSON은 있는데 필요한 PNG/NPZ/NPY/영상이 없으면 부분 손상으로 보고 새 회차에서 다시 생성합니다.

**한 번에 한 정책 프로세스만 실행합니다.** 이전 실행이 끝난 뒤 다음 셀을 실행하고 다른 노트북의 infer/rollout을 동시에 시작하지 않습니다. 실행 중단 후에는 해당 Terminal이나 실행 로그에서 자식 프로세스까지 종료됐는지 확인합니다. `status=completed`는 절차 완료, rollout의 `success=true`는 환경 과제 성공이며 inspect/infer의 `success=false`는 아직 작업을 하지 않았다는 뜻입니다.


## MuJoCo 화면 보기: 로컬·원격 공통 noVNC

<!-- procedure:novnc-1744bd:start -->
![noVNC의 정상 대기와 완료](../교재그림/절차/novnc.svg)

03의 준비 검사와 05의 저장 배열 읽기는 새 실험 창을 열지 않습니다. 기존 완료 결과 재사용에도 새 창이 없습니다.
<!-- procedure:novnc-1744bd:end -->


JupyterLab과 **개인 시뮬레이션 화면을 브라우저 탭 두 개로 열어둡니다.** [README의 최초 접속 안내](README.md#novnc-접속)를 따르세요. 기본 배정은 사용자 A: Jupyter 8890 / 화면 8990, 사용자 B: Jupyter 8891 / 화면 8991입니다. 본인 배정 포트를 사용하며 원격은 두 포트의 SSH 터널을 유지합니다. noVNC에서 Connect를 누르고 암호 `nvidia`를 입력하세요. 처음에는 빈 바탕 화면이 정상이며 실험을 실행하면 MuJoCo 창이 나타납니다.

**이 노트북에서는 로봇이 실제로 행동하는 장면을 봅니다.** 추론 중에는 환경이 대기하고, ACTION EXECUTION에서 예측한 행동을 적용합니다. 스텝 수와 추론 횟수의 변화를 함께 관찰하세요.

설정 셀의 `GUI`는 자동으로 켜집니다. 창 없이 시간 비교를 할 때는 `GUI = False`로 바꾸고 설정 셀을 다시 실행합니다. CLI 대체 실행은 `run_episode.py`에 `--gui`를 붙입니다. 노트북과 CLI를 동시에 실행하지 않습니다.

환경 초기화가 끝난 뒤 실제 초기 장면에서 창이 열립니다. 초기화 중 오류는 노트북 로그에서 확인합니다. 화면에는 실행 이름·모드·현재 상태·스텝·추론 횟수·적용 길이·직전 추론 초가 표시됩니다.

| 화면 상태 | 의미 |
|---|---|
| `INITIAL OBSERVATION` | 초기 관측 준비 완료 |
| `MODEL LOADING` | 모델 로드 중, 환경은 대기 |
| `INFERENCE` | 행동 계산 중, 환경은 대기 |
| `ACTION EXECUTION` | 예측 명령을 환경에 적용 중 |
| `FINISHED` | 종료 이유와 성공 여부 표시. 마지막 장면이며 더 움직이지 않음 |
| `INCOMPLETE / ERROR` | 미완료. 노트북의 원인 로그 확인 |

**정상 종료 화면을 관찰한 뒤 창을 닫으면 셀이 끝납니다.** 결과와 영상은 종료 화면 대기 전에 저장됩니다. 실행 도중 창을 닫으면 다음 안전한 처리 지점에서 사용자 중단으로 기록합니다. GPU 추론·모델 로딩 중에는 그 작업이 반환될 때까지 중단 반영이 늦을 수 있습니다. 창 조작은 표시용 복사본에만 적용되어 정책 카메라와 실제 실험 상태를 바꾸지 않습니다.

**기존 완료 결과를 읽으면 새 GUI는 열리지 않습니다.** 새 시연은 사용하지 않은 실행 이름으로 수행하고 변수·파일 경로도 함께 맞춥니다. GUI 사용 여부는 `result.json`의 `gui_enabled`에 남습니다. 화면 표시 비용이 있으므로 GUI 실행과 비GUI 실행의 시간을 같은 조건처럼 비교하지 않습니다.


**브라우저 탭을 닫아도 실험은 종료되지 않습니다.** 다시 같은 주소로 접속하면 진행 중 화면으로 돌아옵니다. FINISHED 후 화면 안의 MuJoCo 창을 닫아야 셀이 끝납니다. noVNC 연결과 탭은 다음 실험까지 유지하세요. 다른 학생 화면이 보이면 포트를 확인합니다.


## 흐름 그림

![예측한 나머지 40개를 이어 쓰지 않고, 앞 10개 실행 후 새 관측으로 다시 예측합니다.](../교재그림/vla-horizon.svg)

예측한 나머지 40개를 이어 쓰지 않고, 앞 10개 실행 후 새 관측으로 다시 예측합니다.


In [ ]:
# 제공된 실행 셀입니다. 출력과 파일을 확인하며 위에서 아래로 실행합니다.
from pathlib import Path
import sys, subprocess, json, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '10.VLA_v2').is_dir())
SCRIPTS = ROOT / '10.VLA_v2/scripts'
sys.path.insert(0, str(SCRIPTS))
from course_runtime import DATA, lesson_runs, run_or_read_lesson
# 04~08 모두 같은 회차를 사용합니다. 전체 비교를 새로 할 때 session-02 등으로 함께 변경하세요.
GUI = os.environ.get('COURSE_VLA_GUI', '0') == '1'  # 개인 noVNC 화면 사용
SESSION = 'v2-session-01'
RUNS = lesson_runs(DATA, SESSION)
RUNS.mkdir(parents=True, exist_ok=True)
def run_lesson(mode, name, seed=7, execute=10, steps=280, initial_state=0):
    return run_or_read_lesson(SCRIPTS, RUNS, mode, name, seed, execute, steps, initial_state, gui=GUI)


## 예제


### 실행 전 확인 — FINISHED 후 MuJoCo 창 닫기

**noVNC에서 `FINISHED`를 확인하면 화면 안의 MuJoCo 창 오른쪽 위 ×를 눌러 닫아주세요.** 창을 닫아야 아래 셀이 완료되고 다음 단계로 진행됩니다. 결과는 이미 저장되었지만, 마지막 장면을 관찰하도록 창을 유지하므로 노트북의 `[*]`와 경과 시간은 계속 표시됩니다. **브라우저 탭만 닫으면 셀은 끝나지 않습니다.** 실행 중 창을 닫으면 중단으로 처리되므로 `FINISHED`를 확인한 뒤 닫습니다.

행동 실행이 끝난 뒤 완료 창을 닫으면 노트북에 저장 동영상과 실행 결과가 표시됩니다. 이후 비교 실험에서도 같은 순서로 진행합니다.

기존 완료 결과를 재사용하거나 `GUI=False`로 실행하면 새 GUI 창이 열리지 않으며, 창 닫기를 기다리지 않습니다.


In [ ]:
# 제공된 실행 셀입니다. 출력과 파일을 확인하며 위에서 아래로 실행합니다.
# rollout만 행동을 환경에 적용합니다. 같은 과제·seed·초기상태에서 한 조건씩 비교합니다.
result = run_lesson('rollout', 'seed7-h10', execute=10)
from IPython.display import Video, display
display(Video(url='/files/' + (RUNS/'seed7-h10/rollout.mp4').relative_to(ROOT).as_posix(), embed=False))
print(result['status'], result['success'], result['steps'], result['termination_reason'])


### 입력과 실행 흐름 확인


In [ ]:
# 제공된 실행 셀입니다. 출력과 파일을 확인하며 위에서 아래로 실행합니다.
def execution_plan(max_steps, execute):
    return [min(execute, max_steps-start) for start in range(0,max_steps,execute)]
print('23스텝, horizon10:', execution_plan(23,10))
assert execution_plan(23,10) == [10,10,3]


### 코드 해설 — 입력부터 결과까지

**한 조건만 변경:** 기준 seed7-h10과 비교 실행 seed7-h5는 과제·초기 배치·seed·모델을 같게 두고 execute만 10→5로 바꿉니다. 출력 이름도 바꿔 기준 결과를 보존합니다.

```text
관측 A → 미래 50개 예측 [1..10 | 11................50]
                         실행       미실행
                            ↓ 새 관측 B
         새 50개 예측 [1..10 | 11................50]
                         실행       미실행
```

기존 나머지 40개를 계속 이어 쓰지 않고 새 관측으로 바꿉니다. execute=5이면 더 자주 다시 보지만 정책 호출 비용도 늘어날 수 있습니다. 남은 23스텝을 최대 10개씩 쓰면 [10,10,3]이며 마지막 3개가 모자란 묶음이라는 오류는 아닙니다.

**확인 질문:** 더 자주 재관측해도 성공이 반드시 늘지 않는 이유를 자신의 영상·시간 결과에서 찾아보세요.

**적용·재확인:** 자동 출력과 실제 파일의 조건을 함께 확인합니다. 완료된 동일 조건 결과는 다시 읽고, 새 실험은 새 실행 이름을 사용합니다. `SESSION`을 바꾸면 후속 노트북도 맞춥니다. 형상 검사·실행 완료·작업 성공을 따로 확인하며, 예시 수치를 자기 측정값으로 복사하지 않습니다.


## 실행 실습 — execute=10 / 5, 초기0·seed7 고정

seed7-h10과 seed7-h5를 차례로 실행해 영상·스텝·추론 횟수·시간을 비교합니다. 한 조건 한 번으로 우열을 일반화하지 않습니다.

새 실행은 noVNC에서 FINISHED를 확인한 뒤 MuJoCo 창을 닫습니다. 같은 조건의 완료 기록을 재사용하면 새 창이 열리지 않습니다.


In [ ]:
# 기준 실행과 같은 초기0·seed7·최대280스텝에서 execute만 5로 바꿉니다.
# rollout만 행동을 환경에 적용합니다. 같은 과제·seed·초기상태에서 한 조건씩 비교합니다.
comparison = run_lesson('rollout', 'seed7-h5', execute=5)
from IPython.display import Video, display, Markdown
lines = ['실행 | 상태 | 작업 성공 | 스텝 | 추론 횟수 | 실행 초', '--- | --- | --- | --- | --- | ---']
for name, row in [('seed7-h10', result), ('seed7-h5', comparison)]:
    lines.append(f"{name} | {row['status']} | {row['success']} | {row['steps']} | {len(row['chunk_seconds'])} | {row.get('execution_seconds', '미기록')}")
    print(name)
    display(Video(url='/files/' + (RUNS/name/'rollout.mp4').relative_to(ROOT).as_posix(), embed=False))
display(Markdown('\n'.join(lines)))
print('horizon 외 조건을 고정한 두 결과입니다. 한 번의 성공/실패를 일반 성능으로 해석하지 않습니다.')


## 결과 해설과 다음 단계

밖의 반복문은 재관측과 추론, 안의 반복문은 행동 묶음 일부의 적용입니다.

execute는 1~50 범위입니다. 같은 과제·seed·초기상태·최대280스텝을 유지해야 horizon의 영향을 비교할 수 있습니다. 0 벡터를 실물 로봇의 정지로 가정하지 않습니다.

horizon 비교에 초기 배치 변경 한 조건을 더해 평가표를 만듭니다.

출력 표·그래프·영상과 강사 설명을 함께 확인합니다. 별도 관찰 답안을 작성하지 않습니다.
